# 05 — Publication Outputs

Generates manuscript-ready figures (300 DPI PNG + vector PDF, Mermaid sources for Figures 1-2) and exports Tables 1-5 as CSV + Markdown + LaTeX. Writes `docs/figure_table_captions.md`. Ends with a short summary printed from the real computed results.

**Data source policy:** reads only already-computed files under `outputs/tables/` and `outputs/predictions/` — no model or calibrator is refit, no prediction is recomputed, no value is simulated or fabricated.

In [ ]:
import sys
sys.path.append('..')

import shutil
from pathlib import Path

import pandas as pd

from src.config import load_config
from src.reproducibility import set_global_seed, get_logger, log_run_metadata
from src.preprocess import load_cohort_features, drop_duplicate_rows, patient_level_split, check_patient_overlap
from src.plots import plot_reliability_diagram_with_histogram
from src.plot_publication_figures import (
    make_paper_study_flowchart_figure, generate_paper_study_workflow_mermaid,
    make_paper_architecture_diagram_figure, generate_paper_architecture_mermaid,
    export_table_with_metadata,
)

config = load_config()
seed = config["project"]["random_seed"]
set_global_seed(seed)
logger = get_logger(log_file=config["logging"]["log_file"])
logger.info("05_publication_outputs started (seed=%d)", seed)

pp_config = config["paper_publication"]
pe_config = config["paper_final_evaluation"]
ID_COL = config["preprocessing"]["id_column"]
TARGET_COL = config["preprocessing"]["target_column"]
FIGURES_DIR = Path(pp_config["figures_output_dir"])
TABLES_DIR = Path(pp_config["tables_output_dir"])
MERMAID_DIR = Path(pp_config["mermaid_output_dir"])
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Figure 1: study flowchart (real cohort-flow + split counts)

In [ ]:
cohort_flow_df = pd.read_csv(TABLES_DIR / "cohort_flow_counts.csv")

raw_df = load_cohort_features(config=config)
dedup_df = drop_duplicate_rows(raw_df)
train_df, val_df, test_df = patient_level_split(
    dedup_df, id_col=ID_COL, target_col=TARGET_COL,
    train_size=config["preprocessing"]["train_size"], val_size=config["preprocessing"]["val_size"],
    test_size=config["preprocessing"]["test_size"], seed=seed,
)
check_patient_overlap(train_df, val_df, test_df, id_col=ID_COL)

stage_labels = [f"{row['stage']}\nn = {int(row['n'])}" for _, row in cohort_flow_df.iterrows()]
stage_labels += [
    f"Training set (70%)\nn = {len(train_df)}",
    f"Validation set (10%)\nn = {len(val_df)}",
    f"Held-out test set (20%)\nn = {len(test_df)}",
]

fig1_paths = make_paper_study_flowchart_figure(stage_labels, config)
fig1_mermaid = generate_paper_study_workflow_mermaid(stage_labels, MERMAID_DIR / "figure1_study_workflow.mmd")
logger.info("Figure 1 saved: %s, mermaid: %s", fig1_paths, fig1_mermaid)
print("Figure 1:", fig1_paths, fig1_mermaid)

## Figure 2: confidence-calibrated clinical AI architecture

In [ ]:
fig2_paths = make_paper_architecture_diagram_figure(config)
fig2_mermaid = generate_paper_architecture_mermaid(MERMAID_DIR / "figure2_system_architecture.mmd")
logger.info("Figure 2 saved: %s, mermaid: %s", fig2_paths, fig2_mermaid)
print("Figure 2:", fig2_paths, fig2_mermaid)

## Figure 3: final reliability diagrams for the best models

"Best" = top-N (model, calibration_method) pairs by test-set AUROC, from `table_4_main_results.csv`.

In [ ]:
table4 = pd.read_csv(pe_config["main_results_table_path"])
test_predictions_df = pd.read_parquet(pe_config["test_predictions_path"])

top_n = pp_config["best_models_for_reliability"]
best_combos = table4.sort_values("auroc", ascending=False).head(top_n)
print("Best (model, calibration_method) pairs by test AUROC:")
print(best_combos[["model", "calibration_method", "auroc", "brier_score", "ece"]])

fig3_paths = []
for _, row in best_combos.iterrows():
    subset = test_predictions_df[
        (test_predictions_df["model_name"] == row["model"])
        & (test_predictions_df["calibration_method"] == row["calibration_method"])
    ]
    fig = plot_reliability_diagram_with_histogram(
        subset["true_label"].to_numpy(), subset["predicted_prob"].to_numpy(), n_bins=10,
        title=f"{row['model']} \u2014 {row['calibration_method']} (test set, AUROC={row['auroc']:.3f})",
    )
    base = FIGURES_DIR / f"figure3_reliability_{row['model']}_{row['calibration_method']}"
    fig.savefig(base.with_suffix(".png"), dpi=pp_config["dpi"], bbox_inches="tight")
    fig.savefig(base.with_suffix(".pdf"), bbox_inches="tight")
    import matplotlib.pyplot as plt
    plt.close(fig)
    fig3_paths.append(base)
    logger.info("Figure 3 saved: %s.png / .pdf", base)

print("Figure 3 files:", fig3_paths)

## Figures 4-5: final ROC and precision-recall curves

Already generated in `04_final_evaluation.ipynb` (`roc_curves_final.*`, `pr_curves_final.*`) from the same test predictions used above. Copied here under the manuscript figure numbers rather than recomputed, so Figures 4-5 are guaranteed identical to what was actually evaluated.

In [ ]:
figure_copies = [
    ("roc_curves_final", "figure4_roc_curves"),
    ("pr_curves_final", "figure5_precision_recall_curves"),
]
for src_stem, dst_stem in figure_copies:
    for ext in [".png", ".pdf"]:
        src_path = FIGURES_DIR / f"{src_stem}{ext}"
        dst_path = FIGURES_DIR / f"{dst_stem}{ext}"
        if src_path.exists():
            shutil.copy2(src_path, dst_path)
            logger.info("Copied %s -> %s", src_path, dst_path)
        else:
            print(f"WARNING: {src_path} not found — run 04_final_evaluation.ipynb first.")

print("Figures 4-5 copied under manuscript figure numbers.")

## Export Tables 1-5 (CSV + Markdown + LaTeX, with captions/abbreviations/notes)

In [ ]:
table_paths = {
    "table1_cohort": TABLES_DIR / "table_1_cohort.csv",
    "table2_hyperparameters": TABLES_DIR / "table_2_hyperparameters.csv",
    "table3_calibration_metrics": Path(config["calibration"]["metrics_table_path"]),
    "table4_main_results": Path(pe_config["main_results_table_path"]),
    "table5_clinical_thresholds": Path(pe_config["clinical_thresholds_table_path"]),
}

captions = {
    "table1_cohort": (
        "Table 1. Cohort characteristics, overall and by data split.",
        ["Numeric variables: median [interquartile range]. Categorical variables: n (%).",
         "Splits are patient-level (no patient appears in more than one split)."],
        ["ICU"],
    ),
    "table2_hyperparameters": (
        "Table 2. Selected model hyperparameters and validation-set AUROC.",
        ["Hyperparameters were selected using validation-set AUROC only; the test set was not used."],
        ["AUROC"],
    ),
    "table3_calibration_metrics": (
        "Table 3. Calibration metrics for each model x calibration method (validation set).",
        ["Calibrators (Platt, isotonic) were fit and evaluated on the same validation set "
         "(in-sample calibration quality); held-out assessment is in Table 4."],
        ["ECE", "MCE", "NLL"],
    ),
    "table4_main_results": (
        "Table 4. Main results: discrimination and calibration performance on the held-out test set, "
        "with 95% bootstrap confidence intervals for AUROC, Brier score, and ECE.",
        ["95% confidence intervals computed via 1000 stratified bootstrap resamples "
         "(resampled within each outcome class, preserving observed class counts).",
         "Evaluated once on the held-out test set."],
        ["AUROC", "AUPRC", "ECE", "CI", "PPV", "NPV"],
    ),
    "table5_clinical_thresholds": (
        "Table 5. Sensitivity, specificity, PPV, NPV, F1, and balanced accuracy at selected "
        "clinical decision thresholds, held-out test set.",
        ["Thresholds evaluated: " + ", ".join(str(t) for t in sorted({pe_config['decision_threshold'], 0.5})) + "."],
        ["PPV", "NPV"],
    ),
}

table_exports = {}
for key, path in table_paths.items():
    if not path.exists():
        print(f"WARNING: {path} not found \u2014 skipping {key} (run the corresponding notebook first).")
        continue
    df = pd.read_csv(path)
    caption, notes, abbrevs = captions[key]
    table_exports[key] = export_table_with_metadata(
        df, TABLES_DIR / key, caption=caption, notes=notes, abbreviations_used=abbrevs,
    )
    logger.info("Exported %s -> %s", key, table_exports[key])

for k, v in table_exports.items():
    print(k, "->", {fmt: str(p) for fmt, p in v.items()})

## docs/figure_table_captions.md

In [ ]:
captions_lines = [
    "# Figure and Table Captions",
    "",
    "Confidence-Calibrated Machine Learning for Reliable In-Hospital Mortality Prediction.",
    "",
    "## Figures",
    "",
    "**Figure 1.** Study cohort flow, from the total extracted ICU stays through exclusions "
    "to the final analysis cohort and the patient-level train/validation/test split.",
    "",
    "**Figure 2.** Confidence-calibrated clinical AI architecture: EHR features (first 24h of "
    "ICU stay) -> preprocessing -> base models (logistic regression, LightGBM) -> calibration "
    "layer (Platt scaling, isotonic regression) -> calibrated risk score -> clinical decision "
    "support / referral.",
    "",
    "**Figure 3.** Reliability diagrams (with confidence histograms) for the best-performing "
    "model x calibration method combinations by test-set AUROC.",
    "",
    "**Figure 4.** ROC curves for all trained models, calibrated and uncalibrated, on the "
    "held-out test set.",
    "",
    "**Figure 5.** Precision-recall curves for all trained models, calibrated and uncalibrated, "
    "on the held-out test set, with the outcome prevalence baseline shown for reference.",
    "",
    "## Tables",
    "",
    "**Table 1.** Cohort characteristics, overall and by data split.",
    "",
    "**Table 2.** Selected model hyperparameters and validation-set AUROC for each model.",
    "",
    "**Table 3.** Calibration metrics (Brier score, ECE at 10 and 15 bins, MCE, negative "
    "log-likelihood, calibration intercept and slope) for each model x calibration method, "
    "validation set.",
    "",
    "**Table 4.** Main results: discrimination (AUROC, AUPRC) and calibration (Brier score, ECE) "
    "performance on the held-out test set, with 95% bootstrap confidence intervals for AUROC, "
    "Brier score, and ECE (1000 stratified resamples), plus sensitivity/specificity/PPV/NPV/F1/"
    "balanced accuracy at the primary decision threshold.",
    "",
    "**Table 5.** Sensitivity, specificity, PPV, NPV, F1 score, and balanced accuracy at each "
    "evaluated clinical decision threshold, held-out test set.",
    "",
    "## Abbreviations",
    "",
    "- **AUROC**: Area Under the Receiver Operating Characteristic curve",
    "- **AUPRC**: Area Under the Precision-Recall Curve",
    "- **CI**: Confidence Interval",
    "- **ECE**: Expected Calibration Error",
    "- **MCE**: Maximum Calibration Error",
    "- **NLL**: Negative Log-Likelihood",
    "- **PPV**: Positive Predictive Value (Precision)",
    "- **NPV**: Negative Predictive Value",
    "- **ICU**: Intensive Care Unit",
    "- **F1**: harmonic mean of precision and recall (sensitivity)",
    "",
]

captions_path = Path(pp_config["captions_path"])
captions_path.parent.mkdir(parents=True, exist_ok=True)
captions_path.write_text("\n".join(captions_lines), encoding="utf-8")
logger.info("Figure/table captions saved to %s", captions_path)
print(f"Saved {captions_path}")

## Final summary

In [ ]:
n_patients = len(dedup_df)
n_deaths = int(dedup_df[TARGET_COL].sum())

best_by_auroc = table4.loc[table4["auroc"].idxmax()]
best_by_brier = table4.loc[table4["brier_score"].idxmin()]
best_ece_row = table4.loc[table4["ece"].idxmin()]

print("=" * 70)
print("FINAL SUMMARY")
print("=" * 70)
print(f"Number of patients (N): {n_patients}")
print(f"Number of deaths: {n_deaths} ({100 * n_deaths / n_patients:.1f}%)")
print(
    f"Best model+calibration by AUROC: {best_by_auroc['model']} / {best_by_auroc['calibration_method']} "
    f"(AUROC={best_by_auroc['auroc']:.4f}, 95% CI [{best_by_auroc['auroc_ci_lower']:.4f}, "
    f"{best_by_auroc['auroc_ci_upper']:.4f}])"
)
print(
    f"Best model+calibration by Brier score: {best_by_brier['model']} / {best_by_brier['calibration_method']} "
    f"(Brier={best_by_brier['brier_score']:.4f}, 95% CI [{best_by_brier['brier_score_ci_lower']:.4f}, "
    f"{best_by_brier['brier_score_ci_upper']:.4f}])"
)
print(
    f"Best ECE achieved: {best_ece_row['ece']:.4f} "
    f"({best_ece_row['model']} / {best_ece_row['calibration_method']})"
)
print("=" * 70)

log_run_metadata(
    seed=seed,
    extra={
        "notebook": "05_publication_outputs",
        "n_patients": n_patients,
        "n_deaths": n_deaths,
        "best_by_auroc": f"{best_by_auroc['model']}/{best_by_auroc['calibration_method']}",
        "best_by_brier": f"{best_by_brier['model']}/{best_by_brier['calibration_method']}",
        "best_ece": float(best_ece_row["ece"]),
    },
)
logger.info("05_publication_outputs finished")